# CAPSDAC Forecast Printable Graph Codebook

This notebook reads forecast parquet outputs, displays percentage tables, shows graphs inline, and saves PNG/CSV files.


In [1]:

# ============================================================
# CAPSDAC Forecast Codebook: Inline Tables + Printable Graphs
# ============================================================
#
# Purpose:
# - Read the existing forecast parquet outputs
# - Print/display percentage tables directly in the notebook
# - Show graphs inline using plt.show()
# - Save each graph as PNG files locally in the notebook session
# - Optionally copy/save images to lake output folder if supported
#
# Run this AFTER the forecast parquet outputs are created.
# ============================================================

from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt
import os

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

ROOT = "abfss://yzhang@cdewarehouseuser.dfs.core.windows.net/capsdac2_silver_parquet"
OUTPUT_ROOT = f"{ROOT}/ml_outputs/vendor_site_forecast"

FORECAST_DETAIL_OUT = f"{OUTPUT_ROOT}/forecast_5m_vendor_site_program"
FORECAST_TOTAL_OUT = f"{OUTPUT_ROOT}/forecast_summary_total"
FORECAST_VENDOR_OUT = f"{OUTPUT_ROOT}/forecast_summary_vendor"
FORECAST_SITE_OUT = f"{OUTPUT_ROOT}/forecast_summary_site"

# Local notebook image folder
LOCAL_IMG_DIR = "/tmp/capsdac_forecast_printable_graphs"
os.makedirs(LOCAL_IMG_DIR, exist_ok=True)

print("Local image folder:", LOCAL_IMG_DIR)

# ------------------------------------------------------------
# 2. Read forecast parquet outputs
# ------------------------------------------------------------

forecast_5m = spark.read.parquet(FORECAST_DETAIL_OUT)
forecast_summary = spark.read.parquet(FORECAST_TOTAL_OUT)
forecast_vendor_summary = spark.read.parquet(FORECAST_VENDOR_OUT)
forecast_site_summary = spark.read.parquet(FORECAST_SITE_OUT)

print("forecast_5m rows:", forecast_5m.count())
print("forecast_summary rows:", forecast_summary.count())
print("forecast_vendor_summary rows:", forecast_vendor_summary.count())
print("forecast_site_summary rows:", forecast_site_summary.count())

display(forecast_5m.limit(20))
display(forecast_summary.orderBy("ForecastReportMonth"))
display(forecast_vendor_summary.orderBy("ForecastReportMonth"))
display(forecast_site_summary.orderBy("ForecastReportMonth"))

# ------------------------------------------------------------
# 3. Convert to pandas for plotting
# ------------------------------------------------------------

pdf_detail = forecast_5m.toPandas()
pdf_total = forecast_summary.toPandas()
pdf_vendor = forecast_vendor_summary.toPandas()
pdf_site = forecast_site_summary.toPandas()

# Make sure month is printable
for pdf in [pdf_detail, pdf_total, pdf_vendor, pdf_site]:
    if "ForecastReportMonth" in pdf.columns:
        pdf["ForecastReportMonth"] = pdf["ForecastReportMonth"].astype(str)

# ------------------------------------------------------------
# 4. Percentage tables
# ------------------------------------------------------------

# Total by forecast month
total_table = (
    pdf_total
    .sort_values("ForecastReportMonth")
    .copy()
)

print("TABLE 1: Total forecast by month")
display(spark.createDataFrame(total_table))

# Vendor contribution percentage
vendor_total = (
    pdf_vendor
    .groupby(["VendorNumber", "LEAName"], dropna=False)["predicted_vendor_enrollment"]
    .sum()
    .reset_index()
)

vendor_total["vendor_contribution_pct"] = (
    vendor_total["predicted_vendor_enrollment"]
    / vendor_total["predicted_vendor_enrollment"].sum()
    * 100
)

vendor_pct_table = (
    vendor_total
    .sort_values("vendor_contribution_pct", ascending=False)
    .reset_index(drop=True)
)

print("TABLE 2: Vendor contribution percentage")
display(spark.createDataFrame(vendor_pct_table))

# Site contribution percentage
site_total = (
    pdf_site
    .groupby(
        ["VendorNumber", "LEAName", "PreschoolCDSCode", "PreschoolName"],
        dropna=False
    )["predicted_site_enrollment"]
    .sum()
    .reset_index()
)

site_total["site_contribution_pct"] = (
    site_total["predicted_site_enrollment"]
    / site_total["predicted_site_enrollment"].sum()
    * 100
)

site_pct_table = (
    site_total
    .sort_values("site_contribution_pct", ascending=False)
    .reset_index(drop=True)
)

print("TABLE 3: Site contribution percentage")
display(spark.createDataFrame(site_pct_table))

# Monthly vendor contribution percentage
vendor_monthly = pdf_vendor.copy()
vendor_monthly["monthly_total"] = (
    vendor_monthly
    .groupby("ForecastReportMonth")["predicted_vendor_enrollment"]
    .transform("sum")
)
vendor_monthly["monthly_vendor_contribution_pct"] = (
    vendor_monthly["predicted_vendor_enrollment"]
    / vendor_monthly["monthly_total"]
    * 100
)

vendor_monthly_pct_table = (
    vendor_monthly
    .sort_values(
        ["ForecastReportMonth", "monthly_vendor_contribution_pct"],
        ascending=[True, False]
    )
)

print("TABLE 4: Monthly vendor contribution percentage")
display(spark.createDataFrame(vendor_monthly_pct_table))

# Monthly site contribution percentage
site_monthly = pdf_site.copy()
site_monthly["monthly_total"] = (
    site_monthly
    .groupby("ForecastReportMonth")["predicted_site_enrollment"]
    .transform("sum")
)
site_monthly["monthly_site_contribution_pct"] = (
    site_monthly["predicted_site_enrollment"]
    / site_monthly["monthly_total"]
    * 100
)

site_monthly_pct_table = (
    site_monthly
    .sort_values(
        ["ForecastReportMonth", "monthly_site_contribution_pct"],
        ascending=[True, False]
    )
)

print("TABLE 5: Monthly site contribution percentage")
display(spark.createDataFrame(site_monthly_pct_table))

# ------------------------------------------------------------
# 5. Helper for saving and showing charts
# ------------------------------------------------------------

def save_show_chart(filename):
    path = os.path.join(LOCAL_IMG_DIR, filename)
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    print("Saved image:", path)
    plt.show()
    plt.close()

# ------------------------------------------------------------
# 6. Graph 1: Total forecast trend curve
# ------------------------------------------------------------

plt.figure(figsize=(12, 6))
plt.plot(
    total_table["ForecastReportMonth"],
    total_table["predicted_total_enrollment"],
    marker="o"
)
plt.title("Total Forecast Enrollment Trend")
plt.xlabel("Forecast Report Month")
plt.ylabel("Predicted Total Enrollment")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

save_show_chart("figure1_total_forecast_trend_curve.png")

# ------------------------------------------------------------
# 7. Graph 2: Top 10 vendors latest month
# ------------------------------------------------------------

latest_month = pdf_vendor["ForecastReportMonth"].max()

top10_vendor = (
    pdf_vendor[pdf_vendor["ForecastReportMonth"] == latest_month]
    .sort_values("predicted_vendor_enrollment", ascending=False)
    .head(10)
    .copy()
)

top10_vendor["vendor_label"] = (
    top10_vendor["VendorNumber"].astype(str)
    + " - "
    + top10_vendor["LEAName"].astype(str)
)

print("TABLE 6: Top 10 vendors, latest forecast month")
display(spark.createDataFrame(top10_vendor))

plt.figure(figsize=(14, 7))
plt.bar(
    top10_vendor["vendor_label"],
    top10_vendor["predicted_vendor_enrollment"]
)
plt.title(f"Top 10 Vendors by Forecast Enrollment ({latest_month})")
plt.xlabel("VendorNumber - LEAName")
plt.ylabel("Predicted Vendor Enrollment")
plt.xticks(rotation=75, ha="right")

save_show_chart("figure2_top10_vendors.png")

# ------------------------------------------------------------
# 8. Graph 3: Top 20 sites latest month
# ------------------------------------------------------------

top20_site = (
    pdf_site[pdf_site["ForecastReportMonth"] == latest_month]
    .sort_values("predicted_site_enrollment", ascending=False)
    .head(20)
    .copy()
)

top20_site["site_label"] = (
    top20_site["PreschoolCDSCode"].astype(str)
    + " - "
    + top20_site["PreschoolName"].astype(str)
)

print("TABLE 7: Top 20 sites, latest forecast month")
display(spark.createDataFrame(top20_site))

plt.figure(figsize=(16, 8))
plt.bar(
    top20_site["site_label"],
    top20_site["predicted_site_enrollment"]
)
plt.title(f"Top 20 Sites by Forecast Enrollment ({latest_month})")
plt.xlabel("PreschoolCDSCode - PreschoolName")
plt.ylabel("Predicted Site Enrollment")
plt.xticks(rotation=85, ha="right")

save_show_chart("figure3_top20_sites.png")

# ------------------------------------------------------------
# 9. Graph 4: Vendor contribution percentage
# ------------------------------------------------------------

top_vendor_pct = vendor_pct_table.head(15).copy()
top_vendor_pct["vendor_label"] = (
    top_vendor_pct["VendorNumber"].astype(str)
    + " - "
    + top_vendor_pct["LEAName"].astype(str)
)

plt.figure(figsize=(14, 7))
plt.bar(
    top_vendor_pct["vendor_label"],
    top_vendor_pct["vendor_contribution_pct"]
)
plt.title("Top Vendor Contribution Percentage")
plt.xlabel("VendorNumber - LEAName")
plt.ylabel("Contribution %")
plt.xticks(rotation=75, ha="right")

save_show_chart("figure4_vendor_contribution_pct.png")

# ------------------------------------------------------------
# 10. Graph 5: Site contribution percentage
# ------------------------------------------------------------

top_site_pct = site_pct_table.head(20).copy()
top_site_pct["site_label"] = (
    top_site_pct["PreschoolCDSCode"].astype(str)
    + " - "
    + top_site_pct["PreschoolName"].astype(str)
)

plt.figure(figsize=(16, 8))
plt.bar(
    top_site_pct["site_label"],
    top_site_pct["site_contribution_pct"]
)
plt.title("Top Site Contribution Percentage")
plt.xlabel("PreschoolCDSCode - PreschoolName")
plt.ylabel("Contribution %")
plt.xticks(rotation=85, ha="right")

save_show_chart("figure5_site_contribution_pct.png")

# ------------------------------------------------------------
# 11. Graph 6: Vendor share trend curve
# ------------------------------------------------------------

top_vendor_names = (
    vendor_pct_table.head(10)["LEAName"].tolist()
)

vendor_trend = (
    vendor_monthly_pct_table[
        vendor_monthly_pct_table["LEAName"].isin(top_vendor_names)
    ]
    .copy()
)

plt.figure(figsize=(14, 7))

for lea in top_vendor_names:
    one = vendor_trend[vendor_trend["LEAName"] == lea].sort_values("ForecastReportMonth")
    plt.plot(
        one["ForecastReportMonth"],
        one["monthly_vendor_contribution_pct"],
        marker="o",
        label=lea
    )

plt.title("Top Vendor Monthly Share Trend")
plt.xlabel("Forecast Report Month")
plt.ylabel("Monthly Contribution %")
plt.xticks(rotation=45)
plt.legend(loc="best", fontsize=8)
plt.grid(True, alpha=0.3)

save_show_chart("figure6_vendor_share_trend.png")

# ------------------------------------------------------------
# 12. Graph 7: Site heatmap
# ------------------------------------------------------------

top_heatmap_sites = (
    site_pct_table.head(20)["PreschoolName"].tolist()
)

heatmap_df = (
    site_monthly_pct_table[
        site_monthly_pct_table["PreschoolName"].isin(top_heatmap_sites)
    ]
    .pivot_table(
        index="PreschoolName",
        columns="ForecastReportMonth",
        values="monthly_site_contribution_pct",
        aggfunc="sum"
    )
    .fillna(0)
)

plt.figure(figsize=(14, 10))
plt.imshow(heatmap_df.values, aspect="auto")
plt.colorbar(label="Monthly Site Contribution %")
plt.title("Top 20 Site Monthly Contribution Heatmap")
plt.xlabel("Forecast Report Month")
plt.ylabel("Preschool Name")

plt.xticks(
    range(len(heatmap_df.columns)),
    heatmap_df.columns,
    rotation=45
)
plt.yticks(
    range(len(heatmap_df.index)),
    heatmap_df.index
)

save_show_chart("figure7_site_contribution_heatmap.png")

# ------------------------------------------------------------
# 13. Save percentage tables as CSV locally
# ------------------------------------------------------------

total_table.to_csv(os.path.join(LOCAL_IMG_DIR, "table1_total_forecast.csv"), index=False)
vendor_pct_table.to_csv(os.path.join(LOCAL_IMG_DIR, "table2_vendor_contribution_pct.csv"), index=False)
site_pct_table.to_csv(os.path.join(LOCAL_IMG_DIR, "table3_site_contribution_pct.csv"), index=False)
vendor_monthly_pct_table.to_csv(os.path.join(LOCAL_IMG_DIR, "table4_monthly_vendor_contribution_pct.csv"), index=False)
site_monthly_pct_table.to_csv(os.path.join(LOCAL_IMG_DIR, "table5_monthly_site_contribution_pct.csv"), index=False)

print("CSV tables saved locally to:", LOCAL_IMG_DIR)

# ------------------------------------------------------------
# 14. Optional: copy local images/tables to Lakehouse/ADLS if mssparkutils is available
# ------------------------------------------------------------

try:
    LAKE_CHART_OUT = f"{OUTPUT_ROOT}/final_charts_outputs/printable_codebook_exports"

    for file_name in os.listdir(LOCAL_IMG_DIR):
        local_file = os.path.join(LOCAL_IMG_DIR, file_name)
        lake_file = f"{LAKE_CHART_OUT}/{file_name}"

        mssparkutils.fs.cp(
            f"file:{local_file}",
            lake_file,
            True
        )

    print("Copied images and tables to:", LAKE_CHART_OUT)

except Exception as e:
    print("Optional lake copy skipped or failed.")
    print(str(e))

print("DONE: Graphs displayed inline and images saved.")


Local image folder: /tmp/capsdac_forecast_printable_graphs


NameError: name 'spark' is not defined

## Additional Growth Charts

The original charts and code above are unchanged. The cells below add two printable growth charts:

1. Top LEA enrollment growth
2. Top preschool enrollment growth


In [ ]:
# ------------------------------------------------------------
# 15. NEW CHARTS ONLY: Top enrollment growth of LEAs and Preschools
# ------------------------------------------------------------
# This section keeps all original charts/code above unchanged.
# It adds two printable growth charts using the first and latest forecast months:
#   - Figure 8: Top LEA enrollment growth
#   - Figure 9: Top Preschool enrollment growth
# ------------------------------------------------------------

import numpy as np


def first_non_missing(series):
    """Return first non-null value from a pandas Series, otherwise blank."""
    non_missing = series.dropna()
    if len(non_missing) == 0:
        return ""
    return non_missing.iloc[0]


def build_growth_table(
    pdf,
    id_col,
    name_col,
    value_col,
    label_col,
    top_n=10
):
    """Build growth table from earliest forecast month to latest forecast month."""
    required_cols = {"ForecastReportMonth", id_col, value_col}
    missing_cols = required_cols - set(pdf.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns for growth chart: {missing_cols}")

    work = pdf.copy()
    work["ForecastReportMonth"] = work["ForecastReportMonth"].astype(str)

    if name_col not in work.columns:
        work[name_col] = ""

    months = sorted(work["ForecastReportMonth"].dropna().unique().tolist())
    if len(months) < 2:
        raise ValueError("At least two ForecastReportMonth values are needed to calculate growth.")

    first_month = months[0]
    last_month = months[-1]

    grouped = (
        work[work["ForecastReportMonth"].isin([first_month, last_month])]
        .groupby([id_col, name_col, "ForecastReportMonth"], dropna=False)[value_col]
        .sum()
        .reset_index()
    )

    pivot = (
        grouped
        .pivot_table(
            index=[id_col, name_col],
            columns="ForecastReportMonth",
            values=value_col,
            aggfunc="sum",
            fill_value=0
        )
        .reset_index()
    )

    pivot["start_month"] = first_month
    pivot["end_month"] = last_month
    pivot["start_enrollment"] = pivot[first_month]
    pivot["end_enrollment"] = pivot[last_month]
    pivot["enrollment_growth"] = pivot["end_enrollment"] - pivot["start_enrollment"]
    pivot["growth_pct"] = np.where(
        pivot["start_enrollment"] > 0,
        pivot["enrollment_growth"] / pivot["start_enrollment"] * 100,
        np.nan
    )

    pivot[label_col] = np.where(
        pivot[name_col].astype(str).str.strip().ne("") & pivot[name_col].notna(),
        pivot[name_col].astype(str),
        pivot[id_col].astype(str)
    )

    output_cols = [
        id_col,
        name_col,
        label_col,
        "start_month",
        "end_month",
        "start_enrollment",
        "end_enrollment",
        "enrollment_growth",
        "growth_pct"
    ]

    return (
        pivot[output_cols]
        .sort_values("enrollment_growth", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )


# -----------------------------
# Figure 8: Top LEA growth
# -----------------------------
lea_growth_top10 = build_growth_table(
    pdf=pdf_vendor,
    id_col="VendorNumber",
    name_col="LEAName",
    value_col="predicted_vendor_enrollment",
    label_col="LEA_Label",
    top_n=10
)

print("TABLE 8: Top 10 LEA enrollment growth")
display(spark.createDataFrame(lea_growth_top10))

plt.figure(figsize=(14, 7))
plt.bar(
    lea_growth_top10["LEA_Label"],
    lea_growth_top10["enrollment_growth"]
)
plt.title(
    f"Top 10 LEA Enrollment Growth "
    f"({lea_growth_top10['start_month'].iloc[0]} to {lea_growth_top10['end_month'].iloc[0]})"
)
plt.xlabel("LEA Name")
plt.ylabel("Enrollment Growth")
plt.xticks(rotation=75, ha="right")
plt.grid(axis="y", alpha=0.3)

save_show_chart("figure8_top10_lea_enrollment_growth.png")

# -----------------------------
# Figure 9: Top Preschool growth
# -----------------------------
preschool_growth_top20 = build_growth_table(
    pdf=pdf_site,
    id_col="PreschoolCDSCode",
    name_col="PreschoolName",
    value_col="predicted_site_enrollment",
    label_col="Preschool_Label",
    top_n=20
)

print("TABLE 9: Top 20 preschool enrollment growth")
display(spark.createDataFrame(preschool_growth_top20))

plt.figure(figsize=(16, 8))
plt.bar(
    preschool_growth_top20["Preschool_Label"],
    preschool_growth_top20["enrollment_growth"]
)
plt.title(
    f"Top 20 Preschool Enrollment Growth "
    f"({preschool_growth_top20['start_month'].iloc[0]} to {preschool_growth_top20['end_month'].iloc[0]})"
)
plt.xlabel("Preschool Name")
plt.ylabel("Enrollment Growth")
plt.xticks(rotation=85, ha="right")
plt.grid(axis="y", alpha=0.3)

save_show_chart("figure9_top20_preschool_enrollment_growth.png")

# Save new growth source tables locally
lea_growth_top10.to_csv(
    os.path.join(LOCAL_IMG_DIR, "table8_top10_lea_enrollment_growth.csv"),
    index=False
)
preschool_growth_top20.to_csv(
    os.path.join(LOCAL_IMG_DIR, "table9_top20_preschool_enrollment_growth.csv"),
    index=False
)

print("New growth chart source tables saved locally to:", LOCAL_IMG_DIR)

# Optional: copy only the new growth files to Lakehouse/ADLS if mssparkutils is available
try:
    LAKE_CHART_OUT = f"{OUTPUT_ROOT}/final_charts_outputs/printable_codebook_exports"

    new_growth_files = [
        "figure8_top10_lea_enrollment_growth.png",
        "figure9_top20_preschool_enrollment_growth.png",
        "table8_top10_lea_enrollment_growth.csv",
        "table9_top20_preschool_enrollment_growth.csv"
    ]

    for file_name in new_growth_files:
        local_file = os.path.join(LOCAL_IMG_DIR, file_name)
        lake_file = f"{LAKE_CHART_OUT}/{file_name}"
        mssparkutils.fs.cp(f"file:{local_file}", lake_file, True)

    print("Copied new growth charts and tables to:", LAKE_CHART_OUT)

except Exception as e:
    print("Optional lake copy for new growth charts skipped or failed.")
    print(str(e))

print("DONE: Added top LEA and top preschool enrollment growth charts.")
